# Wav2Lip Lip-Sync on Colab **T4**

Makes a face (video or image) speak an audio clip. Perfect for making a
Wan-animated character talk with your Floyd / Cuffem / Player audio.

### This runs on the Colab T4 GPU only — NOT HuggingFace ZeroGPU.
It clones the open-source Wav2Lip code and runs `inference.py` on the T4
that Colab assigns to *your* runtime. It never calls the hosted HF Space,
so ZeroGPU (HF's shared, quota'd A100 slices) is never involved.

**First:** Runtime -> Change runtime type -> **T4 GPU** -> Save. Then run the
cells top to bottom. Step 0 proves you actually got a T4.


## Step 0 - Prove the GPU is a T4 (not ZeroGPU, not CPU)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.'
name = torch.cuda.get_device_name(0)
print('Torch is using:', name)
print('VERIFIED: running on', name, '- this is Colab hardware, not HF ZeroGPU.')


## Step 1 - Get Wav2Lip (maintained fork) and install deps
Uses `justinjohn0306/Wav2Lip`, which fixes the old-dependency problems and
hosts the model checkpoints.


In [ ]:
import os
if not os.path.isdir('/content/Wav2Lip'):
    !git clone -q https://github.com/justinjohn0306/Wav2Lip /content/Wav2Lip
%cd /content/Wav2Lip
!pip install -q -r requirements.txt
!pip install -q batch-face gdown
print('deps installed')


## Step 2 - Download the model checkpoints into the runtime


In [ ]:
%cd /content/Wav2Lip
import os
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('face_detection/detection/sfd', exist_ok=True)
B='https://github.com/justinjohn0306/Wav2Lip/releases/download/models/'
!wget -q -c $B'wav2lip.pth'     -O checkpoints/wav2lip.pth
!wget -q -c $B'wav2lip_gan.pth' -O checkpoints/wav2lip_gan.pth
!wget -q -c $B's3fd.pth'        -O face_detection/detection/sfd/s3fd.pth
!wget -q -c $B'mobilenet.pth'   -O checkpoints/mobilenet.pth
!wget -q -c 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth' -O checkpoints/GFPGANv1.4.pth
print('checkpoints ready:', os.listdir('checkpoints'))


## Step 3 - Upload your inputs
**FACE** = a video or image of the character (e.g. a Wan clip from
`D:\\MatrixVideos`). **AUDIO** = the voice line (wav/mp3, e.g. floyddeath.wav).


In [ ]:
from google.colab import files
print('Upload the FACE (mp4 / png / jpg):')
FACE = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('Upload the AUDIO (wav / mp3):')
AUDIO = '/content/Wav2Lip/' + list(files.upload().keys())[0]
print('FACE :', FACE)
print('AUDIO:', AUDIO)


## Step 4 - Run Wav2Lip on the T4
`wav2lip_gan.pth` gives the best mouth quality. `--nosmooth` helps on
single faces. Adjust `--pads` (top bottom left right) if the mouth box is off.


In [ ]:
%cd /content/Wav2Lip
!python inference.py \
  --checkpoint_path checkpoints/wav2lip_gan.pth \
  --face "$FACE" --audio "$AUDIO" \
  --outfile /content/result.mp4 \
  --nosmooth --pads 0 10 0 0 --resize_factor 1
print('done -> /content/result.mp4')


## Step 5 - Preview and download the result


In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open('/content/result.mp4','rb').read()).decode()
HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')


In [ ]:
from google.colab import files
files.download('/content/result.mp4')  # saves to your Downloads; move it to D:\\MatrixVideos


---
### Optional: sharper faces with GFPGAN
If the face looks soft, re-run Step 4 adding `--out_height 720` (and keep the
GFPGAN checkpoint from Step 2). Higher = slower on the T4.
